**BERT Fine**-**Tuning**

In [ ]:
#Install & Import
!pip install transformers datasets scikit-learn

import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [5]:
#Load Dataset (IMDB)
dataset = load_dataset("imdb")

train_data = dataset['train']
test_data = dataset['test']

train_data = train_data.select(range(2000))   # reduce train size
test_data = test_data.select(range(1000))     # reduce test size

In [ ]:
#Preprocessing
def clean_text(example):
    example['text'] = example['text'].lower()
    return example

train_data = train_data.map(clean_text)
test_data = test_data.map(clean_text)

In [ ]:
#Tokenization
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length')

train_data = train_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)

In [ ]:
#Model
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    ignore_mismatched_sizes=True
)

In [9]:
#Metrics Function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
#Training
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    do_eval=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
#Evaluation
results = trainer.evaluate()
print(results)

In [ ]:
#Confusion Matrix
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

preds = trainer.predict(test_data)
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

cm = confusion_matrix(y_true, y_pred)

sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

Experiment 1: Freeze BERT

In [ ]:
for param in model.bert.parameters():
    param.requires_grad = False

Experiment 2: Fine-tune Last 2 Layers

In [ ]:
for name, param in model.bert.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False